In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [2]:
# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0

torch.manual_seed(1024)

In [3]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print("length of dataset in characters: ", len(text))

print(text[:100])

length of dataset in characters:  1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [4]:
chars = sorted(list(set(text)))

vocab_size = len(chars)

print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [5]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]             # 將字母編碼成數字
decode = lambda l: ''.join([itos[i] for i in l])    # 將數字解碼回字母

print(encode("hii there"))                          # 示範編碼結果，數字應該介於0~64
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [6]:
data = torch.tensor(encode(text), dtype=torch.long)         # 使用長整數資料型態

print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [24]:
n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

In [25]:
# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data

    idx = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in idx])
    y = torch.stack([data[i+1:i+block_size+1] for i in idx])

    x, y = x.to(device), y.to(device)
    
    return x, y

In [26]:
# 估算loss，這邊利用decorator強制讓函式不計算梯度
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        
        out[split] = losses.mean()
    
    model.train()
    
    return out

In [ ]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)             # K
        self.query = nn.Linear(n_embd, head_size, bias=False)           # Q
        self.value = nn.Linear(n_embd, head_size, bias=False)           # V
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        
        wei = q @ k.transpose(-2,-1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        
        
        out = wei @ v
        return out


class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        
        out = self.dropout(self.proj(out))
        return out

In [ ]:
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
# 將所有材料組裝成Block
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        
        

        self.sa = MultiHeadAttention(n_head, head_size)     # MultiHead Self-Attention
        self.ffwd = FeedFoward(n_embd)                      # FeedFoward
        self.ln1 = nn.LayerNorm(n_embd)                     # LayerNorm
        self.ln2 = nn.LayerNorm(n_embd)                     # LayerNorm

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        
        
        
        return x

In [ ]:
# 設計Block重複多少次
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)                               # 字符的embedding
        self.position_embedding_table = nn.Embedding(block_size, n_embd)                            # 位置的embedding
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])        # 組n_layer層Block
        self.ln_f = nn.LayerNorm(n_embd)                                                            # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)                                                # 最後輸出前的線性轉換n_embd --> vocab_size

    def forward(self, idx, targets=None):
        B, T = idx.shape                                # 會得到(batch_size, block_size)

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))

        x = tok_emb + pos_emb
        
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)         # 計算loss

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [31]:
model = BigramLanguageModel()
m = model.to(device)

# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

0.209729 M parameters


In [32]:
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.4106, val loss 4.4045
step 100: train loss 2.6600, val loss 2.6702
step 200: train loss 2.5079, val loss 2.5133
step 300: train loss 2.4217, val loss 2.4347
step 400: train loss 2.3601, val loss 2.3816
step 500: train loss 2.3115, val loss 2.3258
step 600: train loss 2.2391, val loss 2.2738
step 700: train loss 2.2129, val loss 2.2223
step 800: train loss 2.1663, val loss 2.1985
step 900: train loss 2.1344, val loss 2.1689
step 1000: train loss 2.1030, val loss 2.1429
step 1100: train loss 2.0711, val loss 2.1025
step 1200: train loss 2.0429, val loss 2.0905
step 1300: train loss 2.0179, val loss 2.0818
step 1400: train loss 1.9902, val loss 2.0587
step 1500: train loss 1.9665, val loss 2.0329
step 1600: train loss 1.9483, val loss 2.0412
step 1700: train loss 1.9267, val loss 2.0164
step 1800: train loss 1.9268, val loss 2.0019
step 1900: train loss 1.8978, val loss 1.9913
step 2000: train loss 1.8871, val loss 1.9795
step 2100: train loss 1.8724, val loss 1.9694


In [33]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))



BUCKINGHAM:
Whild tells it thing amone talking uplance.

SICHMONES:
And as no bleat mother coves: than of,
Noing in joy should weselibilts gree
To titin me, milved our raised
Town; thing follow.

HAMIONA:
Let's sanch bees!
Do ever my death your wingers; and so,
trable be voils Parener, and my opizatace!
Are blence, but the iaral; I stay tall bear, be man Ell,
Money father I do cuteen'd had,
Lance, go hounoure's ht.

SICINIUS:
So to it then, and to be mattle
Andings spations, with strad
In been of the plow'd enattersures Cany.

DULIZES:
him inseasure ented her shall, my gace hath go's out:
And hard, the him: his touch
Sirs of thy houst'st of these so you tune were secul,
Tollust, but me life on his, bost aumaboir to chote.

Some:
To, dell witer am in a yield tyral are to
hils languted of
the broaty, thing my says strefores,
in eniunesss unto Kint the aptremblaugh
For ounce of that ast terre
I remotion so with man oful tatice?
Our good, not burn Thy dam, bretther fool ounce?
How all, t